<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/11_treatment_planning/imrtp_optimization_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMRT plan optimization (IMRTP)

## Introduction

This notebook reproduces, in pyCERR, the fluence-map optimization workflow of
MATLAB CERR's
[`IMRTP/optim example/runOptimExample.m`](https://github.com/cerr/CERR/blob/master/IMRTP/optim%20example/runOptimExample.m).

Inverse IMRT planning splits each beam into many **pencil beams (beamlets)** and
finds the beamlet intensities (weights) that best match the prescription. pyCERR
provides:

* `cerr.imrtp.imrtp_problem` - the IMRT problem model (`initIMRTProblem`,
  `addGoal`, `addEquispacedBeams`, `conditionBeam`).
* `cerr.imrtp.dosecalc.generateQIBInfluence` - a QIB (quadrant-infinite-beam,
  Ahnesjo pencil-beam) engine that fills the per-beamlet dose-deposition
  (*influence*) matrices - the analog of MATLAB's `getInfluenceM`.
* `cerr.imrtp.dosecalc.getIMDose` - assembles a 3-D dose from beamlet weights
  (analog of `getIMDose`).

The optimization itself - a **weighted bounded least-squares** problem that
MATLAB solves with `quadprog` - is solved here with
`scipy.optimize.lsq_linear`.


## Install pyCERR

In [ ]:
!pip install "pyCERR @ git+https://github.com/cerr/pyCERR.git"

## 1. Build an example plan

We build a small synthetic CT (a water cylinder) with a spherical **PTV** target
and a **Cord** organ-at-risk. To use real data instead, replace this cell with
`planC = pc.loadDcmDir(r'/path/to/CT_and_RTSTRUCT')`.


In [ ]:
import os, tempfile
import numpy as np
import nibabel as nib
from cerr import plan_container as pc

N, NS = 56, 20                                   # in-plane size, num slices
ct = (np.zeros((N, N, NS)) - 1000).astype(np.int16)        # air = -1000 HU
rr, cc = np.meshgrid(np.arange(N), np.arange(N), indexing='ij')
body = ((rr - N // 2) ** 2 + (cc - N // 2) ** 2) < (N * 0.42) ** 2
for s in range(NS):
    ct[:, :, s][body] = 0                                  # water = 0 HU
ctFile = os.path.join(tempfile.gettempdir(), 'imrtp_demo_ct.nii.gz')
nib.save(nib.Nifti1Image(ct, np.diag([3., 3., 3., 1.])), ctFile)
planC = pc.loadNiiScan(ctFile, imageType='CT SCAN')

rr, cc, ss = np.meshgrid(np.arange(N), np.arange(N), np.arange(NS), indexing='ij')
ptvMask = (rr - N//2)**2 + (cc - N//2)**2 + ((ss - NS//2) * 2.)**2 < 7**2
cordMask = (((rr - (N//2 + 14))**2 + (cc - N//2)**2) < 4**2) & body[..., None]
planC = pc.importStructureMask(ptvMask.astype(float), 0, 'PTV', planC)
planC = pc.importStructureMask(cordMask.astype(float), 0, 'Cord', planC)
print('Structures:', [s.structureName for s in planC.structure])

## 2. Set up the IMRT problem

Add each structure as a **goal** (mark the PTV as the target), then add a set of
equispaced beams and resolve their geometry (`conditionBeam` places the
isocenter at the target centre-of-mass and derives the source position).


In [ ]:
from cerr.imrtp import imrtp_problem as imp
from cerr.imrtp import imrtp
from cerr.dataclasses import structure as structr

im = imp.initIMRTProblem(planC)

gPTV = imp.addGoal(im, 0, planC)          # 0 -> first structure (PTV)
gPTV.isTarget = 'yes'
gPTV.xySampleRate = 1                     # sample every in-plane voxel

gCord = imp.addGoal(im, 1, planC)         # 1 -> second structure (Cord)
gCord.isTarget = 'no'
gCord.xySampleRate = 1

NUM_BEAMS = 5
imp.addEquispacedBeams(im, NUM_BEAMS, 0.0, planC)   # 5 beams from gantry 0 deg
im.params.algorithm = 'QIB'

for b in im.beams:                        # resolve isocenter / source geometry
    imp.conditionBeam(b, im, planC, getattr(b, 'autoFieldsOverride', None))

print('%d beams at gantry angles: %s'
      % (len(im.beams), [b.gantryAngle for b in im.beams]))

## 3. Generate the beamlet influence matrices

`generateQIBInfluence` ray-traces each beam, lays a beamlet grid over the
target's projection, and computes each beamlet's dose contribution to every
(sampled) goal-structure voxel. This is the most expensive step (~tens of
seconds for this small example).


In [ ]:
from cerr.imrtp.dosecalc import generateQIBInfluence, getIMDose

generateQIBInfluence(im, planC)

# Number of pencil beams (beamlets) per beam:
pbCounts = [b.RTOGPBVectorsM.shape[0] if b.beamlets else 0 for b in im.beams]
nPB = int(sum(pbCounts))
print('pencil beams per beam:', pbCounts, '| total beamlets:', nPB)

## 4. Assemble the per-structure influence matrices

For each structure we build a sparse matrix `A` of shape
`(num voxels, num beamlets)` whose entry `A[v, p]` is the dose beamlet `p`
deposits in voxel `v` (the analog of MATLAB's `getInfluenceM`). The beamlet
column order matches the flat weight vector expected by `getIMDose`.


In [ ]:
from scipy.sparse import coo_matrix, vstack

nVox = int(np.prod(planC.scan[0].getScanSize()))
wStart = np.concatenate(([0], np.cumsum(pbCounts)))     # column offset per beam

def influence_matrix(strUID):
    R, C, V = [], [], []
    for j, beam in enumerate(im.beams):
        pbIdx = 0
        for blt in beam.beamlets:
            if blt.strUID != strUID:
                continue
            if blt.indexV is not None and blt.indexV.size:
                R.append(blt.indexV)
                C.append(np.full(blt.indexV.size, wStart[j] + pbIdx))
                V.append(blt.influence)
            pbIdx += 1
    if R:
        R, C, V = np.concatenate(R), np.concatenate(C), np.concatenate(V)
    else:
        R = C = V = np.array([], dtype=int)
    return coo_matrix((V, (R, C)), shape=(nVox, nPB)).tocsr()

A_ptv = influence_matrix(gPTV.strUID)
A_cord = influence_matrix(gCord.strUID)
# keep only the rows (voxels) actually sampled for each structure:
ptvRows = np.unique(A_ptv.nonzero()[0])
cordRows = np.unique(A_cord.nonzero()[0])
A_ptv, A_cord = A_ptv[ptvRows], A_cord[cordRows]
print('PTV influence matrix:', A_ptv.shape, '| Cord:', A_cord.shape)

## 5. Optimize the beamlet weights

We solve the **weighted least-squares** problem (MATLAB's `quadprog` analog):
minimize `w_PTV * ||A_PTV @ w - d_rx||^2 + w_Cord * ||A_Cord @ w||^2`,
driving the PTV toward the prescription `d_rx` while pushing cord dose to zero.
The relative importances trade target coverage against OAR sparing.
`scipy.optimize.lsq_linear` enforces the beamlet bounds `0 <= w <= w_max`.


In [ ]:
from scipy.optimize import lsq_linear

PRESCRIPTION = 70.0          # PTV prescription dose (Gy)
IMP_PTV, IMP_CORD = 1.0, 0.6 # relative importances
W_MAX = 50.0                 # max beamlet intensity

A = vstack([np.sqrt(IMP_PTV) * A_ptv, np.sqrt(IMP_CORD) * A_cord])
b = np.concatenate([np.sqrt(IMP_PTV) * np.full(A_ptv.shape[0], PRESCRIPTION),
                    np.zeros(A_cord.shape[0])])

res = lsq_linear(A, b, bounds=(0, W_MAX), max_iter=80, tol=1e-3, verbose=1)
weights = res.x
print('\nbeamlet weights: min %.2f  max %.2f  mean %.2f'
      % (weights.min(), weights.max(), weights.mean()))

## 6. Compute and store the optimized dose

`getIMDose` combines the beamlets with the optimized weights into a 3-D dose,
which we add to `planC.dose`.


In [ ]:
ptvNum = structr.getStructNumFromUID(gPTV.strUID, planC)
cordNum = structr.getStructNumFromUID(gCord.strUID, planC)

dose3M = getIMDose(im, weights, [ptvNum, cordNum], planC)
doseNum = imrtp.doseToPlanC(dose3M, im, planC, fractionGroupID='IMRTP optimized')

ptvVox = ptvMask.astype(bool)
cordVox = cordMask.astype(bool)
print('PTV  mean/min/max: %.1f / %.1f / %.1f Gy'
      % (dose3M[ptvVox].mean(), dose3M[ptvVox].min(), dose3M[ptvVox].max()))
print('Cord mean/max:     %.1f / %.1f Gy'
      % (dose3M[cordVox].mean(), dose3M[cordVox].max()))

## 7. Visualize the result

The beamlet-weight profile, the dose colourwash on a central axial slice, and
the cumulative dose-volume histogram (`cerr.dvh`).


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 2.6))
plt.bar(np.arange(nPB), weights, width=1.0, color='tab:purple')
plt.xlabel('beamlet index'); plt.ylabel('weight')
plt.title('Optimized beamlet weights'); plt.tight_layout(); plt.show()

In [ ]:
from cerr.contour import rasterseg as rs

xV, yV, zV = planC.scan[0].getScanXYZVals()
k = NS // 2
ctSlc = planC.scan[0].getScanArray()[:, :, k]
doseSlc = dose3M[:, :, k]
ext = [xV[0], xV[-1], yV[-1], yV[0]]

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(ctSlc, cmap='gray', extent=ext)
dm = ax.imshow(np.ma.masked_where(doseSlc <= 0.05 * doseSlc.max(), doseSlc),
               cmap='jet', alpha=0.55, extent=ext)
plt.colorbar(dm, ax=ax, fraction=0.046, label='Dose (Gy)')
for strNum, color in [(ptvNum, 'r'), (cordNum, 'c')]:
    m = rs.getStrMask(strNum, planC)[:, :, k].astype(float)
    ax.contour(xV, yV, m, levels=[0.5], colors=[color], linewidths=1.5)
ax.set_title('Optimized dose - axial slice %d' % k); plt.show()

In [ ]:
from cerr import dvh

plt.figure(figsize=(6, 4))
for strNum, label, color in [(ptvNum, 'PTV', 'tab:red'),
                             (cordNum, 'Cord', 'tab:blue')]:
    dosesV, volsV, _ = dvh.getDVH(strNum, doseNum, planC)
    binWidth = max(float(np.max(dosesV)) / 400.0, 1e-3)
    binsV, histV = dvh.doseHist(dosesV, volsV, binWidth)
    cumPct = 100.0 * np.flip(np.cumsum(np.flip(histV))) / np.sum(histV)
    plt.plot(binsV, cumPct, label=label, color=color, lw=2)
plt.xlabel('Dose (Gy)'); plt.ylabel('Volume (%)'); plt.ylim(0, 105)
plt.grid(alpha=0.3); plt.legend(); plt.title('Cumulative DVH (optimized plan)')
plt.show()

## Notes

* **Engine**: `'QIB'` is a fast analytic pencil-beam engine bundled with pyCERR
  (Ahnesjo kernel tables). Higher-fidelity Monte-Carlo engines (`'DPM'`,
  `'VMC++'`) are placeholders in this build.
* **Goal voxels only**: `getIMDose` fills dose inside the goal structures. Add a
  body/skin structure as a low-importance goal to obtain dose elsewhere.
* **Tuning**: change `PRESCRIPTION`, the `IMP_*` importances, `NUM_BEAMS`, the
  beam start angle, or the per-goal `xySampleRate` to trade coverage, OAR
  sparing, speed and resolution.
* The same problem can be set up and run interactively from the desktop GUI:
  `from cerr.imrtp import IMRTPGui; IMRTPGui(planC)`.
